In [ ]:
# Step 1: Install all required packages and download the language model
!pip install spacy pdfplumber fpdf pandas tqdm
!python -m spacy download en_core_web_sm


In [ ]:
def analyze_specification_document(pdf_path, custom_materials=None):
    print("1. Extracting text from PDF...")
    pages_content = extract_text_from_pdf(pdf_path)
    final_output = []

    print("2. Setting up material matcher...")
    matcher = setup_material_matcher(custom_materials)
    material_mentions = {}

    print("3. Finding and grouping material mentions...")
    for page, text in tqdm(pages_content.items()):
        doc = nlp(text)
        matches = matcher(doc)
        for match_id, start, end in matches:
            name = doc[start:end].text.strip().lower()
            sentence = doc[start:end].sent.text.strip()

            if name not in material_mentions:
                material_mentions[name] = []
            material_mentions[name].append({'page': page, 'sentence': sentence, 'full_page_context': text})

    print("4. Analyzing each material for specific specifications and codes...")
    for name, mentions in tqdm(material_mentions.items()):
        unique_pages = sorted(list(set(m['page'] for m in mentions)))
        full_context = " ".join(m['full_page_context'] for m in mentions)

        # Attempt to find a specific standard specification and code within the sentences
        specific_spec = "No Information Available"
        relevant_codes = set()

        for mention in mentions:
            codes_in_sentence = extract_by_regex(CODE_PATTERNS, mention['sentence'])
            relevant_codes.update(codes_in_sentence)
            specific_spec = mention['sentence']  # Corrected indentation
            break

        if not relevant_codes:
            for mention in mentions:
                relevant_codes.update(extract_by_regex(CODE_PATTERNS, mention['full_page_context']))

        material_type = classify_material_type(name, full_context)

        codes_str = "\n".join(f"- {c}" for c in sorted(list(relevant_codes))) if relevant_codes else "No Information Available"

        final_output.append({
            "Material Name": name.title(),
            "Specific Material Type": material_type,
            "Specific Standard Specification": specific_spec,
            "Code/Standard": codes_str,
            "Any other relevant information": f"Found on pages: {', '.join(map(str, unique_pages))}"
        })

    if not final_output:
        print("No materials found in the document.")
        return pd.DataFrame()

    df = pd.DataFrame(final_output)
    df.insert(0, 'Sl. No.', range(1, 1 + len(df)))
    df = df[['Sl. No.', 'Material Name', 'Specific Material Type', 'Specific Standard Specification', 'Code/Standard', 'Any other relevant information']]
    return df


In [ ]:
# Step 3: Upload the technical specification PDF
print("Please upload the technical specification PDF file...")
uploaded = files.upload()

pdf_path = next(iter(uploaded))
print(f"\n✅ Successfully uploaded '{pdf_path}'")

In [ ]:
# Step 4: Analyzing
def analyze_specification_document(pdf_path, custom_materials=None):
    print("1. Extracting text from PDF...")
    pages_content = extract_text_from_pdf(pdf_path)
    final_output = []

    print("2. Setting up material matcher...")
    matcher = setup_material_matcher(custom_materials)
    material_mentions = {}

    print("3. Finding and grouping material mentions...")
    for page, text in tqdm(pages_content.items()):
        doc = nlp(text)
        matches = matcher(doc)
        for match_id, start, end in matches:
            name = doc[start:end].text.strip().lower()
            sentence = doc[start:end].sent.text.strip()

            if name not in material_mentions:
                material_mentions[name] = []
            material_mentions[name].append({'page': page, 'sentence': sentence, 'full_page_context': text})

    print("4. Analyzing each material for specific specifications and codes...")
    for name, mentions in tqdm(material_mentions.items()):
        unique_pages = sorted(list(set(m['page'] for m in mentions)))
        full_context = " ".join(m['full_page_context'] for m in mentions)

        specific_spec = "No Information Available"
        relevant_codes = set()

        for mention in mentions:
            codes_in_sentence = extract_by_regex(CODE_PATTERNS, mention['sentence'])
            relevant_codes.update(codes_in_sentence)

            if codes_in_sentence:
                 specific_spec = mention['sentence']
                 break

        if not relevant_codes:
             for mention in mentions:
                  relevant_codes.update(extract_by_regex(CODE_PATTERNS, mention['full_page_context']))


        material_type = classify_material_type(name, full_context)


        codes_str = "\n".join(f"- {c}" for c in sorted(list(relevant_codes))) if relevant_codes else "No Information Available"


        final_output.append({
            "Material Name": name.title(),
            "Specific Material Type": material_type,
            "Specific Standard Specification": specific_spec,
            "Code/Standard": codes_str,
            "Any other relevant information": f"Found on pages: {', '.join(map(str, unique_pages))}"
        })

    if not final_output:
        print("No materials found in the document.")
        return pd.DataFrame()

    df = pd.DataFrame(final_output)
    df.insert(0, 'Sl. No.', range(1, 1 + len(df)))
    # Reorder columns to match desired output structure, including the new field
    df = df[['Sl. No.', 'Material Name', 'Specific Material Type', 'Specific Standard Specification', 'Code/Standard', 'Any other relevant information']]
    return df


# Prompt for custom material terms to search for
custom_input = input(" Enter any material names (comma-separated), or press Enter to use defaults: ")
custom_materials = [term.strip() for term in custom_input.split(",") if term.strip()]

# Run main analysis function
print("\n Starting analysis with enhanced extraction... this may take a moment.")
df = analyze_specification_document(pdf_path, custom_materials)

if not df.empty:
    # Preview results
    print("\n Analysis Complete. Preview of Extracted Data:")
    display(df.head())

    # Generate and Download PDF Report
    print("\n Generating PDF report...")
    export_to_pdf(df)
    files.download("Extracted_Technical_Spec_Report.pdf")

    # Generate and Download CSV Report
    print("\n Generating CSV report...")
    df_cleaned = clean_csv_for_export(df)
    csv_output_path = "Extracted_Technical_Spec_Report.csv"
    df_cleaned.to_csv(csv_output_path, index=False)
    files.download(csv_output_path)
else:
    print("\n Analysis finished, but no data was extracted to generate reports.")

In [ ]:
# Step 5: Evaluate the accuracy against a ground truth file
from google.colab import files
uploaded = files.upload()

# Use the actual uploaded file name
ground_truth_filename = next(iter(uploaded)) # Get the filename from the uploaded dictionary
ground_truth_df = pd.read_excel(ground_truth_filename, engine='openpyxl')

# Evaluate
print("\n--- Comparing extracted data with ground truth... ---")
precision, recall, f1 = compute_extraction_precision_recall(df, ground_truth_df)

print(f"\n✅ Evaluation Metrics:")
print(f"Precision: {precision:.2%}")
print(f"   Recall: {recall:.2%}")
print(f" F1 Score: {f1:.2%}")